# GPU RF Shared-Book ML Trading

Train collapsed-label CUDA random forests on all available pre-2020 Quant Warehouse data, then trade the 2020+ out-of-sample predictions with one shared multi-asset book.

The strategy does not rebalance to top-k every day. It holds positions until an opposite signal exits them, then fills open slots from the best current opportunities that pass the `> 0.5` score filter. Position size is fixed at `1 / top_k`, so unused slots remain cash.

The notebook compares long-only, short-only, and shared long+short variants for `top_k = [5, 10, 20, 40]`. All execution paths consume the same shared-book target weights. The framework rows are cost/accounting adapters over the same weights because the existing native framework examples in this repo are single-symbol sleeve runners and are not valid for this shared-book strategy.

In [1]:
from __future__ import annotations

from pathlib import Path
from time import perf_counter
import random
import sys
import warnings

import numpy as np
import pandas as pd

from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report, f1_score
from sklearn.preprocessing import LabelEncoder

QUANT_WAREHOUSE_ROOT = Path('/home/jlee153232/PycharmProjects/quant-warehouse')
if str(QUANT_WAREHOUSE_ROOT) not in sys.path:
    sys.path.insert(0, str(QUANT_WAREHOUSE_ROOT))

import cupy as cp
import cudf
from cuml.ensemble import RandomForestClassifier as CuRandomForestClassifier

from quant_warehouse.platforms.data_providers.fmp.target_engineering.event_pairs import EventPairStore
from quant_warehouse.research_tools import (
    BinaryTargetConfig,
    FamilyEvaluationConfig,
    build_collapsed_bullish_event_target_panel,
    build_fundamental_feature_panel,
    build_oracle_trade_target_panel,
    cap_features_by_quality,
    load_fmp_event_pairs,
    screen_fmp_equity_universe,
)
from quant_warehouse.warehouse.api import Warehouse

pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 180)
pd.set_option('display.float_format', lambda value: f'{value:,.4f}')
warnings.filterwarnings('ignore', category=FutureWarning)

print('cupy', cp.__version__, 'cuda_devices', cp.cuda.runtime.getDeviceCount())
print('cudf', cudf.__version__)

cupy 14.1.1 cuda_devices 1
cudf 26.06.00


In [2]:
RANDOM_SEED = 20260702
MIN_MARKET_CAP = 1_000_000_000_000
START_DATE = '1900-01-01'
END_DATE = None
TRAIN_END = pd.Timestamp('2019-12-31')
OOS_START = pd.Timestamp('2020-01-01')

TOP_K_VALUES = [5, 10, 20, 40]
ENTRY_THRESHOLD = 0.50
EXIT_THRESHOLD = 0.50
MIN_FEATURE_COVERAGE = 0.50
MAX_FEATURES_PER_FAMILY = 50
MIN_TRAIN_ROWS_PER_FAMILY = 250
MIN_CLASSES_PER_FAMILY = 2
CAPITAL_BASE = 1_000_000.0

# Approximate realistic all-in trading frictions. These are applied to notional turnover.
FRAMEWORK_COST_MODELS = {
    'backtesting_py_shared_book': {'commission_bps': 1.0, 'slippage_bps': 5.0},
    'zipline_shared_book': {'commission_bps': 0.5, 'slippage_bps': 5.0},
    'nautilus_shared_book': {'commission_bps': 1.0, 'slippage_bps': 7.5},
}

RF_PARAMS = {
    'n_estimators': 300,
    'max_depth': 16,
    'max_features': 'sqrt',
    'n_bins': 128,
    'random_state': RANDOM_SEED,
    'n_streams': 8,
}

FEATURE_CONFIG = FamilyEvaluationConfig(
    provider='fmp',
    market_cap_min=MIN_MARKET_CAP,
    start_date=START_DATE,
    end_date=END_DATE,
    max_features_per_family=MAX_FEATURES_PER_FAMILY,
)
TARGET_CONFIG = BinaryTargetConfig(
    provider='fmp',
    start_date=START_DATE,
    end_date=END_DATE,
    event_families=('congress', 'insider', 'analyst_rating', 'price_target', 'earnings'),
    oracle_trade_k_by_frequency={'YE': tuple(range(1, 13))},
)

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
FEATURE_CONFIG, TARGET_CONFIG

(FamilyEvaluationConfig(provider='fmp', market_cap_min=1000000000000, country='US', exchanges=('NASDAQ', 'NYSE', 'AMEX'), screen_limit=5000, start_date='1900-01-01', end_date=None, filing_lag_days=45, horizons=(20, 60, 120), min_observations=120, max_features_per_family=50),
 BinaryTargetConfig(provider='fmp', start_date='1900-01-01', end_date=None, event_families=('congress', 'insider', 'analyst_rating', 'price_target', 'earnings'), oracle_trade_k_by_frequency={'YE': (1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12)}, oracle_trade_min_profit_pct=0.01, oracle_trade_long_entry_price_col='high', oracle_trade_long_exit_price_col='low', oracle_trade_short_entry_price_col='low', oracle_trade_short_exit_price_col='high', event_alignment_tolerance_days=7, collapsed_bullish_event_types=('congress_buy', 'insider_buy', 'analyst_upgrade', 'price_target_raise', 'earnings_beat')))

## Build Data

Training uses every available row before 2020. Out-of-sample classification metrics and trading results both use 2020+ rows.

In [3]:
started = perf_counter()
WAREHOUSE = Warehouse()
EVENT_STORE = EventPairStore(backend=WAREHOUSE.backend, catalog=WAREHOUSE.catalog)

symbols, raw_universe, universe_eligibility, universe_source = screen_fmp_equity_universe(FEATURE_CONFIG, warehouse=WAREHOUSE)
print({'universe_source': universe_source, 'eligible_symbols': len(symbols)})
display(universe_eligibility.loc[universe_eligibility['eligible']].head(30))

{'universe_source': 'openbb:fmp', 'eligible_symbols': 14}


,symbol,eligible,reason,screen_market_cap
0,NVDA,True,ok,4785585180000
1,GOOGL,True,ok,4368788098535
2,GOOG,True,ok,4349493623926
3,AAPL,True,ok,4323663859280
4,MSFT,True,ok,2854597080400
5,AMZN,True,ok,2599991070000
6,SPCX,True,ok,2059971841733
7,AVGO,True,ok,1757164597200
8,TSLA,True,ok,1597307716000
9,META,True,ok,1555825021126


In [4]:
raw_feature_panel, raw_feature_metadata, feature_diagnostics, feature_timings = build_fundamental_feature_panel(
    symbols, FEATURE_CONFIG, warehouse=WAREHOUSE
)
selected_features, selected_feature_metadata, feature_quality = cap_features_by_quality(
    raw_feature_panel, raw_feature_metadata, max_features=MAX_FEATURES_PER_FAMILY
)
feature_panel = raw_feature_panel[['symbol', 'date', *selected_features]].copy()
feature_panel['symbol'] = feature_panel['symbol'].astype(str).str.upper()
feature_panel['date'] = pd.to_datetime(feature_panel['date'], errors='coerce').dt.normalize()
print({
    'raw_feature_panel_rows': len(raw_feature_panel),
    'selected_feature_columns': len(selected_features),
    'selected_feature_families': selected_feature_metadata['family'].nunique(),
    **feature_timings,
})
display(selected_feature_metadata.groupby(['source', 'family'], as_index=False).agg(feature_count=('feature', 'nunique')).sort_values(['source', 'family']))

{'raw_feature_panel_rows': 100125, 'selected_feature_columns': 365, 'selected_feature_families': 15, 'raw_panel_build_seconds': 1.4927786230109632}


,source,family,feature_count
0,financetoolkit,ft_growth_balance,50
1,financetoolkit,ft_growth_cash,50
2,financetoolkit,ft_growth_income,38
3,financetoolkit,ft_ratios_efficiency,5
4,financetoolkit,ft_ratios_liquidity,7
5,financetoolkit,ft_ratios_profitability,14
6,financetoolkit,ft_ratios_solvency,15
7,financetoolkit,ft_ratios_valuation,24
8,fmp,fmp_balance_mcap,50
9,fmp,fmp_cash_mcap,39


In [5]:
events, event_diagnostics, event_load_seconds = load_fmp_event_pairs(
    symbols, TARGET_CONFIG, event_store=EVENT_STORE, include_historical=True
)
event_symbols = tuple(event_diagnostics.loc[event_diagnostics['combined_rows'].gt(0), 'symbol'].sort_values())
feature_panel = feature_panel.loc[feature_panel['symbol'].isin(event_symbols)].copy()

collapsed_event_panel, collapsed_event_metadata = build_collapsed_bullish_event_target_panel(
    feature_panel[['symbol', 'date']], events, TARGET_CONFIG
)
oracle_panel, oracle_metadata, oracle_seconds = build_oracle_trade_target_panel(
    event_symbols, TARGET_CONFIG, warehouse=WAREHOUSE
)
print({
    'event_symbols': len(event_symbols),
    'event_rows': len(events),
    'event_load_seconds': round(event_load_seconds, 3),
    'collapsed_event_rows': len(collapsed_event_panel),
    'oracle_rows': len(oracle_panel),
    'oracle_columns': len(oracle_metadata),
    'oracle_seconds': round(oracle_seconds, 3),
})
display(event_diagnostics.sort_values('combined_rows', ascending=False).head(20))

{'event_symbols': 13, 'event_rows': 5836, 'event_load_seconds': 0.911, 'collapsed_event_rows': 100114, 'oracle_rows': 100114, 'oracle_columns': 36, 'oracle_seconds': 4.334}


,symbol,cached_rows,historical_rows,combined_rows,event_families
9,MSFT,65,971,1036,"(analyst_rating, congress, earnings, insider, ..."
0,AAPL,62,885,947,"(analyst_rating, congress, earnings, insider, ..."
11,NVDA,47,812,859,"(analyst_rating, congress, earnings, insider, ..."
1,AMZN,66,742,808,"(analyst_rating, congress, earnings, insider, ..."
6,GOOGL,60,590,650,"(analyst_rating, congress, earnings, insider, ..."
13,TSLA,60,459,519,"(analyst_rating, congress, earnings, insider, ..."
8,META,55,437,492,"(analyst_rating, congress, earnings, insider, ..."
10,MU,0,121,121,"(earnings,)"
7,LLY,0,104,104,"(earnings,)"
3,BRK-A,0,102,102,"(earnings,)"


In [6]:
def collapsed_label_rows(collapsed_event_panel: pd.DataFrame, oracle_panel: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    event_target = 'target_event_collapsed__bullish'
    event_activity = f'_target_activity__{event_target}'
    frames = []
    diagnostics = []

    event_frame = collapsed_event_panel.copy()
    if event_target in event_frame.columns:
        activity = pd.to_numeric(event_frame.get(event_activity, 0), errors='coerce').fillna(0).astype('int8')
        value = pd.to_numeric(event_frame[event_target], errors='coerce').fillna(0).astype('int8')
        bullish = event_frame.loc[value.eq(1), ['symbol', 'date']].copy()
        bullish['collapsed_label'] = 'event_bullish'
        bullish['label_source'] = 'event_collapsed'
        frames.append(bullish)
        diagnostics.append({'source': 'event_collapsed', 'candidate_rows': int(activity.gt(0).sum()), 'used_rows': len(bullish), 'dropped_rows': int(activity.gt(0).sum()) - len(bullish), 'note': 'mirror/non-bullish event rows excluded'})

    long_cols = sorted(c for c in oracle_panel.columns if c.startswith('target_oracle_trade_entry__') and c.endswith('_long'))
    short_cols = sorted(c for c in oracle_panel.columns if c.startswith('target_oracle_trade_entry__') and c.endswith('_short'))
    if long_cols and short_cols:
        oracle = oracle_panel[['symbol', 'date', *long_cols, *short_cols]].copy()
        long_any = oracle[long_cols].apply(pd.to_numeric, errors='coerce').fillna(0).gt(0).any(axis=1)
        short_any = oracle[short_cols].apply(pd.to_numeric, errors='coerce').fillna(0).gt(0).any(axis=1)
        ambiguous = long_any & short_any
        long_rows = oracle.loc[long_any & ~short_any, ['symbol', 'date']].copy()
        long_rows['collapsed_label'] = 'oracle_long'
        long_rows['label_source'] = 'oracle_trade'
        short_rows = oracle.loc[short_any & ~long_any, ['symbol', 'date']].copy()
        short_rows['collapsed_label'] = 'oracle_short'
        short_rows['label_source'] = 'oracle_trade'
        frames.extend([long_rows, short_rows])
        diagnostics.append({'source': 'oracle_trade', 'candidate_rows': int((long_any | short_any).sum()), 'used_rows': len(long_rows) + len(short_rows), 'dropped_rows': int(ambiguous.sum()), 'note': 'ambiguous long+short rows dropped after k collapse'})

    labels = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=['symbol', 'date', 'collapsed_label', 'label_source'])
    labels['symbol'] = labels['symbol'].astype(str).str.upper()
    labels['date'] = pd.to_datetime(labels['date'], errors='coerce').dt.normalize()
    labels = labels.dropna(subset=['symbol', 'date', 'collapsed_label']).drop_duplicates()
    return labels.sort_values(['date', 'symbol', 'collapsed_label']).reset_index(drop=True), pd.DataFrame(diagnostics)

label_rows, label_diagnostics = collapsed_label_rows(collapsed_event_panel, oracle_panel)
print({'collapsed_label_rows': len(label_rows), 'classes': sorted(label_rows['collapsed_label'].unique())})
display(label_diagnostics)
display(label_rows.groupby(['label_source', 'collapsed_label'], as_index=False).agg(rows=('symbol', 'size'), symbols=('symbol', 'nunique'), min_date=('date', 'min'), max_date=('date', 'max')))

{'collapsed_label_rows': 11581, 'classes': ['event_bullish', 'oracle_long', 'oracle_short']}


,source,candidate_rows,used_rows,dropped_rows,note
0,event_collapsed,4131,2353,1778,mirror/non-bullish event rows excluded
1,oracle_trade,9228,9228,0,ambiguous long+short rows dropped after k coll...


,label_source,collapsed_label,rows,symbols,min_date,max_date
0,event_collapsed,event_bullish,2353,13,1993-03-26,2026-06-22
1,oracle_trade,oracle_long,4677,13,1970-07-17,2026-06-12
2,oracle_trade,oracle_short,4551,13,1970-07-09,2026-06-22


## Train GPU RF Family Models Pre-2020

Each feature family gets one CUDA random forest. Out-of-sample metrics are measured on 2020+ labeled rows.

In [7]:
def prepare_family_dataset(feature_panel, feature_metadata, labels, *, source, family, min_feature_coverage):
    family_meta = feature_metadata.loc[feature_metadata['source'].astype(str).eq(source) & feature_metadata['family'].astype(str).eq(family)]
    features = [f for f in family_meta['feature'].drop_duplicates().tolist() if f in feature_panel.columns]
    numeric_features = [f for f in features if pd.to_numeric(feature_panel[f], errors='coerce').notna().any()]
    if not numeric_features:
        return pd.DataFrame(), []
    merged = labels.merge(feature_panel[['symbol', 'date', *numeric_features]], on=['symbol', 'date'], how='inner')
    if merged.empty:
        return merged, numeric_features
    numeric = merged[numeric_features].apply(pd.to_numeric, errors='coerce')
    coverage = numeric.notna().mean(axis=1)
    merged = merged.loc[coverage.ge(min_feature_coverage)].copy()
    if merged.empty:
        return merged, numeric_features
    numeric = numeric.loc[merged.index]
    medians = numeric.median(axis=0).replace([np.inf, -np.inf], np.nan).fillna(0.0)
    merged[numeric_features] = numeric.replace([np.inf, -np.inf], np.nan).fillna(medians).astype('float32')
    return merged.reset_index(drop=True), numeric_features

def fit_gpu_random_forest(train, features):
    encoder = LabelEncoder()
    y = encoder.fit_transform(train['collapsed_label'].astype(str))
    model = CuRandomForestClassifier(**RF_PARAMS)
    model.fit(cudf.from_pandas(train[features].astype('float32')), cudf.Series(y.astype('int32')))
    return model, encoder

def predict_proba_frame(model, encoder, frame, features):
    proba = model.predict_proba(cudf.from_pandas(frame[features].astype('float32')))
    proba_np = proba.to_numpy() if hasattr(proba, 'to_numpy') else cp.asnumpy(proba)
    out = pd.DataFrame(proba_np, columns=[f'prob__{label}' for label in encoder.classes_], index=frame.index)
    for label in ['event_bullish', 'oracle_long', 'oracle_short']:
        col = f'prob__{label}'
        if col not in out.columns:
            out[col] = 0.0
    return out

def score_classifier(model, encoder, frame, features):
    proba = predict_proba_frame(model, encoder, frame, features)
    y_true = frame['collapsed_label'].astype(str).to_numpy()
    y_pred = encoder.inverse_transform(np.asarray(proba[[f'prob__{label}' for label in encoder.classes_]].to_numpy().argmax(axis=1)).astype(int))
    return {
        'rows': len(frame),
        'accuracy': accuracy_score(y_true, y_pred),
        'balanced_accuracy': balanced_accuracy_score(y_true, y_pred),
        'macro_f1': f1_score(y_true, y_pred, average='macro', zero_division=0),
    }

model_rows = []
models = {}
train_started = perf_counter()
for source, family in selected_feature_metadata[['source', 'family']].drop_duplicates().sort_values(['source', 'family']).itertuples(index=False, name=None):
    family_frame, features = prepare_family_dataset(feature_panel, selected_feature_metadata, label_rows, source=str(source), family=str(family), min_feature_coverage=MIN_FEATURE_COVERAGE)
    if family_frame.empty:
        model_rows.append({'source': source, 'family': family, 'status': 'skipped_empty', 'features': len(features), 'rows': 0})
        continue
    train = family_frame.loc[pd.to_datetime(family_frame['date']).le(TRAIN_END)].copy()
    oos = family_frame.loc[pd.to_datetime(family_frame['date']).ge(OOS_START)].copy()
    if len(train) < MIN_TRAIN_ROWS_PER_FAMILY or train['collapsed_label'].nunique() < MIN_CLASSES_PER_FAMILY:
        model_rows.append({'source': source, 'family': family, 'status': 'skipped_sparse_train', 'features': len(features), 'rows': len(family_frame), 'train_rows': len(train), 'oos_rows': len(oos), 'train_classes': train['collapsed_label'].nunique()})
        continue
    fit_started = perf_counter()
    model, encoder = fit_gpu_random_forest(train, features)
    fit_seconds = perf_counter() - fit_started
    models[(source, family)] = {'model': model, 'encoder': encoder, 'features': features}
    train_scores = score_classifier(model, encoder, train, features)
    oos_scores = score_classifier(model, encoder, oos, features) if not oos.empty and oos['collapsed_label'].nunique() > 1 else {'rows': len(oos), 'accuracy': np.nan, 'balanced_accuracy': np.nan, 'macro_f1': np.nan}
    model_rows.append({'source': source, 'family': family, 'status': 'ok', 'features': len(features), 'rows': len(family_frame), 'train_rows': len(train), 'oos_rows': len(oos), 'classes': family_frame['collapsed_label'].nunique(), 'fit_seconds': fit_seconds, **{f'train_{k}': v for k, v in train_scores.items()}, **{f'oos_{k}': v for k, v in oos_scores.items()}})

model_results = pd.DataFrame(model_rows).sort_values(['status', 'oos_macro_f1', 'oos_balanced_accuracy'], ascending=[True, False, False]).reset_index(drop=True)
print({'trained_models': len(models), 'elapsed_seconds': round(perf_counter() - train_started, 3)})
display(model_results)

[20:53:13] /__w/nvforest/nvforest/python/nvforest/build/cp311-abi3-linux_aarch64/_deps/treelite-src/src/serializer.cc:202: The model you are loading originated from a newer Treelite version; some functionalities may be unavailable.
Currently running Treelite version 4.6.1
The model checkpoint was generated from Treelite version 4.7.0


[20:53:15] /__w/nvforest/nvforest/python/nvforest/build/cp311-abi3-linux_aarch64/_deps/treelite-src/src/serializer.cc:202: The model you are loading originated from a newer Treelite version; some functionalities may be unavailable.
Currently running Treelite version 4.6.1
The model checkpoint was generated from Treelite version 4.7.0


[20:53:16] /__w/nvforest/nvforest/python/nvforest/build/cp311-abi3-linux_aarch64/_deps/treelite-src/src/serializer.cc:202: The model you are loading originated from a newer Treelite version; some functionalities may be unavailable.
Currently running Treelite version 4.6.1
The model checkpoint was generated from Treelite version 4.7.0


[20:53:18] /__w/nvforest/nvforest/python/nvforest/build/cp311-abi3-linux_aarch64/_deps/treelite-src/src/serializer.cc:202: The model you are loading originated from a newer Treelite version; some functionalities may be unavailable.
Currently running Treelite version 4.6.1
The model checkpoint was generated from Treelite version 4.7.0


[20:53:20] /__w/nvforest/nvforest/python/nvforest/build/cp311-abi3-linux_aarch64/_deps/treelite-src/src/serializer.cc:202: The model you are loading originated from a newer Treelite version; some functionalities may be unavailable.
Currently running Treelite version 4.6.1
The model checkpoint was generated from Treelite version 4.7.0


...

[20:53:35] /__w/nvforest/nvforest/python/nvforest/build/cp311-abi3-linux_aarch64/_deps/treelite-src/src/serializer.cc:202: The model you are loading originated from a newer Treelite version; some functionalities may be unavailable.
Currently running Treelite version 4.6.1
The model checkpoint was generated from Treelite version 4.7.0


[20:53:36] /__w/nvforest/nvforest/python/nvforest/build/cp311-abi3-linux_aarch64/_deps/treelite-src/src/serializer.cc:202: The model you are loading originated from a newer Treelite version; some functionalities may be unavailable.
Currently running Treelite version 4.6.1
The model checkpoint was generated from Treelite version 4.7.0


{'trained_models': 15, 'elapsed_seconds': 27.826}


[20:53:38] /__w/nvforest/nvforest/python/nvforest/build/cp311-abi3-linux_aarch64/_deps/treelite-src/src/serializer.cc:202: The model you are loading originated from a newer Treelite version; some functionalities may be unavailable.
Currently running Treelite version 4.6.1
The model checkpoint was generated from Treelite version 4.7.0


,source,family,status,features,rows,train_rows,oos_rows,classes,fit_seconds,train_accuracy,train_balanced_accuracy,train_macro_f1,oos_accuracy,oos_balanced_accuracy,oos_macro_f1
0,financetoolkit,ft_ratios_valuation,ok,24,10885,7126,3759,3,1.6556,0.5265,0.4494,0.4688,0.4302,0.3953,0.3912
1,financetoolkit,ft_ratios_solvency,ok,15,10885,7126,3759,3,1.6130,0.5236,0.4464,0.4660,0.2934,0.3448,0.2676
2,fmp,fmp_cash_mcap,ok,39,10430,6671,3759,3,1.6215,0.9319,0.8437,0.8741,0.2969,0.3642,0.2555
3,financetoolkit,ft_ratios_efficiency,ok,5,10885,7126,3759,3,1.5784,0.5240,0.4456,0.4654,0.2854,0.3410,0.2515
4,financetoolkit,ft_ratios_profitability,ok,14,10885,7126,3759,3,1.6129,0.5261,0.4464,0.4659,0.2825,0.3402,0.2496
5,fmp,fmp_daily_ev_yield,ok,7,10885,7126,3759,3,1.5812,0.9145,0.8062,0.8417,0.2876,0.3533,0.2486
6,fmp,fmp_daily_mcap_yield,ok,14,10826,7067,3759,3,1.6463,0.9530,0.8749,0.9026,0.2961,0.3635,0.2480
7,fmp,fmp_balance_mcap,ok,50,10826,7067,3759,3,1.6049,0.9458,0.8676,0.8971,0.2985,0.3607,0.2423
8,fmp,fmp_income_mcap,ok,31,10885,7126,3759,3,1.6926,0.9499,0.8769,0.9026,0.2966,0.3588,0.2413
9,fmp,fmp_daily_ev_multiple,ok,7,10822,7063,3759,3,1.5803,0.9216,0.8145,0.8504,0.2857,0.3524,0.2413


## Inference Scores

Scores are averaged across feature-family models. `long_score = P(event_bullish) + P(oracle_long)`, capped at 1. `short_score = P(oracle_short)`.

In [8]:
def prepare_prediction_frame(feature_panel, features, min_feature_coverage):
    base_cols = ['symbol', 'date']
    numeric = feature_panel[features].apply(pd.to_numeric, errors='coerce')
    coverage = numeric.notna().mean(axis=1)
    out = feature_panel.loc[coverage.ge(min_feature_coverage), base_cols].copy()
    if out.empty:
        return out
    numeric = numeric.loc[out.index]
    medians = numeric.median(axis=0).replace([np.inf, -np.inf], np.nan).fillna(0.0)
    out[features] = numeric.replace([np.inf, -np.inf], np.nan).fillna(medians).astype('float32')
    return out.reset_index(drop=True)

pred_frames = []
for (source, family), payload in models.items():
    pred_input = prepare_prediction_frame(feature_panel, payload['features'], MIN_FEATURE_COVERAGE)
    if pred_input.empty:
        continue
    proba = predict_proba_frame(payload['model'], payload['encoder'], pred_input, payload['features'])
    scored = pred_input[['symbol', 'date']].copy()
    scored['source'] = source
    scored['family'] = family
    scored['long_score_component'] = (proba['prob__event_bullish'] + proba['prob__oracle_long']).clip(0, 1).to_numpy()
    scored['short_score_component'] = proba['prob__oracle_short'].clip(0, 1).to_numpy()
    pred_frames.append(scored)

predictions_by_family = pd.concat(pred_frames, ignore_index=True) if pred_frames else pd.DataFrame()
mean_scores = (
    predictions_by_family.groupby(['symbol', 'date'], as_index=False)
    .agg(long_score=('long_score_component', 'mean'), short_score=('short_score_component', 'mean'), model_count=('family', 'nunique'))
)
mean_scores = mean_scores.loc[pd.to_datetime(mean_scores['date']).ge(OOS_START)].copy()
mean_scores['net_score'] = mean_scores['long_score'] - mean_scores['short_score']
print({'prediction_rows': len(mean_scores), 'symbols': mean_scores['symbol'].nunique(), 'dates': mean_scores['date'].nunique()})
display(mean_scores[['long_score', 'short_score', 'net_score', 'model_count']].describe())

{'prediction_rows': 21151, 'symbols': 13, 'dates': 1627}


,long_score,short_score,net_score,model_count
count,"21,151.0000","21,151.0000","21,151.0000","21,151.0000"
mean,0.5349,0.4651,0.0698,15.0000
std,0.0564,0.0564,0.1128,0.0000
min,0.2304,0.1793,-0.5392,15.0000
25%,0.5039,0.4327,0.0078,15.0000
50%,0.5405,0.4595,0.0811,15.0000
75%,0.5673,0.4961,0.1346,15.0000
max,0.8207,0.7696,0.6413,15.0000


In [9]:
def load_close_prices(symbols):
    frames = []
    for symbol in symbols:
        prices = WAREHOUSE.read_prices(symbol, provider=FEATURE_CONFIG.provider, start=START_DATE, end=END_DATE)
        if prices is None or prices.empty or 'close' not in prices.columns:
            continue
        frame = prices[['close']].copy()
        frame['symbol'] = symbol
        frame['date'] = pd.to_datetime(frame.index, errors='coerce').normalize()
        frames.append(frame.reset_index(drop=True))
    prices = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=['symbol', 'date', 'close'])
    prices['symbol'] = prices['symbol'].astype(str).str.upper()
    prices['date'] = pd.to_datetime(prices['date'], errors='coerce').dt.normalize()
    return prices.dropna(subset=['symbol', 'date', 'close']).drop_duplicates(['symbol', 'date'])

close_prices = load_close_prices(event_symbols)
wide_close = close_prices.pivot(index='date', columns='symbol', values='close').sort_index().ffill()
next_returns = wide_close.pct_change().shift(-1)
print({'price_symbols': wide_close.shape[1], 'price_dates': wide_close.shape[0], 'oos_dates': int((wide_close.index >= OOS_START).sum())})
display(wide_close.tail())

{'price_symbols': 13, 'price_dates': 14109, 'oos_dates': 1627}


symbol,AAPL,AMZN,AVGO,BRK-A,BRK-B,GOOG,GOOGL,LLY,META,MSFT,MU,NVDA,TSLA
date,,,,,,,,,,,,,
2026-06-17,295.9500,237.5000,392.9000,"737,300.0000",491.2800,362.1000,363.7900,"1,112.0000",567.5800,378.9100,"1,043.1900",204.6500,396.3800
2026-06-18,298.0100,244.3900,411.3500,"733,609.7700",489.4600,367.4600,368.0300,"1,098.5700",577.2200,379.4000,"1,133.9900",210.6900,400.4900
2026-06-22,297.0100,232.7900,392.1300,"734,399.9900",488.6900,348.7800,349.6800,"1,102.0800",563.8500,367.3400,"1,211.3800",208.6500,405.0500
2026-06-23,294.3000,234.1100,380.1500,"737,800.0500",492.8100,346.0800,346.1300,"1,107.0800",562.2000,373.9400,"1,051.7700",200.0400,381.6100
2026-06-24,293.0800,234.2700,382.0700,"742,900.0000",494.8100,345.0400,345.2900,"1,117.2600",557.6700,365.4600,"1,048.5100",199.0000,375.5300


## Shared-Book Trading Policy

For every day: exit held positions on the opposite score, then fill open slots from the best current candidates above the entry threshold. Long+short uses one shared `top_k` capacity across both sides.

In [10]:
def build_shared_book_weights(scores, symbols, dates, *, top_k, variant, entry_threshold=0.5, exit_threshold=0.5):
    by_date = {pd.Timestamp(d): g.set_index('symbol') for d, g in scores.groupby('date')}
    positions: dict[str, int] = {}
    rows = []
    events = []
    for date in dates:
        date = pd.Timestamp(date)
        day = by_date.get(date)
        if day is None or day.empty:
            rows.extend({'date': date, 'symbol': s, 'target_weight': positions.get(s, 0) / top_k} for s in symbols)
            continue
        # exits first
        for symbol, side in list(positions.items()):
            if symbol not in day.index:
                continue
            long_score = float(day.at[symbol, 'long_score'])
            short_score = float(day.at[symbol, 'short_score'])
            if side == 1 and short_score > exit_threshold:
                del positions[symbol]
                events.append({'date': date, 'symbol': symbol, 'action': 'exit_long', 'score': short_score})
            elif side == -1 and long_score > exit_threshold:
                del positions[symbol]
                events.append({'date': date, 'symbol': symbol, 'action': 'exit_short', 'score': long_score})
        open_slots = max(0, int(top_k) - len(positions))
        candidates = []
        if open_slots > 0:
            for symbol, row in day.iterrows():
                if symbol in positions:
                    continue
                long_score = float(row['long_score'])
                short_score = float(row['short_score'])
                if variant == 'long_only':
                    if long_score > entry_threshold:
                        candidates.append((long_score, symbol, 1, 'enter_long'))
                elif variant == 'short_only':
                    if short_score > entry_threshold:
                        candidates.append((short_score, symbol, -1, 'enter_short'))
                elif variant == 'long_short':
                    best_side = 1 if long_score >= short_score else -1
                    best_score = max(long_score, short_score)
                    if best_score > entry_threshold:
                        candidates.append((best_score, symbol, best_side, 'enter_long' if best_side == 1 else 'enter_short'))
                else:
                    raise ValueError(f'unknown variant {variant}')
            candidates.sort(key=lambda x: x[0], reverse=True)
            for score, symbol, side, action in candidates[:open_slots]:
                positions[symbol] = side
                events.append({'date': date, 'symbol': symbol, 'action': action, 'score': score})
        rows.extend({'date': date, 'symbol': s, 'target_weight': positions.get(s, 0) / top_k} for s in symbols)
    weights = pd.DataFrame(rows).pivot(index='date', columns='symbol', values='target_weight').reindex(index=dates, columns=symbols).fillna(0.0)
    return weights, pd.DataFrame(events)

def performance_metrics(returns, equity, weights, trades, *, framework, variant, top_k, cost_bps):
    clean = returns.dropna()
    years = len(clean) / 252 if len(clean) else np.nan
    total_return = float(equity.iloc[-1] / equity.iloc[0] - 1) if len(equity) > 1 else np.nan
    ann_return = float((1 + total_return) ** (1 / years) - 1) if years and years > 0 and total_return > -1 else np.nan
    ann_vol = float(clean.std() * np.sqrt(252)) if len(clean) else np.nan
    sharpe = float(clean.mean() / clean.std() * np.sqrt(252)) if len(clean) and clean.std() else np.nan
    drawdown = equity / equity.cummax() - 1
    gross = weights.abs().sum(axis=1)
    net = weights.sum(axis=1)
    return {
        'framework': framework, 'variant': variant, 'top_k': top_k, 'cost_bps': cost_bps,
        'days': len(clean), 'trades': len(trades), 'final_equity': float(equity.iloc[-1]),
        'total_return': total_return, 'annualized_return': ann_return, 'annualized_vol': ann_vol, 'sharpe': sharpe,
        'max_drawdown': float(drawdown.min()), 'win_rate': float(clean.gt(0).mean()) if len(clean) else np.nan,
        'avg_gross_exposure': float(gross.mean()), 'avg_net_exposure': float(net.mean()),
        'fully_invested_days': float(gross.ge(0.999).mean()), 'cash_days': float(gross.eq(0).mean()),
    }

def run_weight_backtest(weights, next_returns, *, cost_bps, capital_base=1_000_000.0):
    aligned_returns = next_returns.reindex(index=weights.index, columns=weights.columns).fillna(0.0)
    prev_weights = weights.shift(1).fillna(0.0)
    turnover = weights.sub(prev_weights).abs().sum(axis=1)
    gross_returns = (weights * aligned_returns).sum(axis=1)
    costs = turnover * (cost_bps / 10_000.0)
    net_returns = gross_returns - costs
    equity = capital_base * (1 + net_returns.fillna(0.0)).cumprod()
    equity.name = 'portfolio_value'
    return net_returns, equity, turnover

oos_dates = pd.DatetimeIndex(sorted(set(mean_scores['date']).intersection(next_returns.index)))
oos_dates = oos_dates[oos_dates >= OOS_START]
trade_symbols = tuple(sorted(set(mean_scores['symbol']).intersection(next_returns.columns)))
print({'trade_symbols': len(trade_symbols), 'oos_dates': len(oos_dates)})

{'trade_symbols': 13, 'oos_dates': 1627}


In [11]:
weight_artifacts = {}
trade_logs = []
summary_rows = []
for variant in ['long_only', 'short_only', 'long_short']:
    for top_k in TOP_K_VALUES:
        weights, trades = build_shared_book_weights(mean_scores, trade_symbols, oos_dates, top_k=top_k, variant=variant, entry_threshold=ENTRY_THRESHOLD, exit_threshold=EXIT_THRESHOLD)
        weight_artifacts[(variant, top_k)] = weights
        trades = trades.assign(variant=variant, top_k=top_k)
        trade_logs.append(trades)
        for framework, costs in FRAMEWORK_COST_MODELS.items():
            cost_bps = float(costs['commission_bps'] + costs['slippage_bps'])
            rets, equity, turnover = run_weight_backtest(weights, next_returns, cost_bps=cost_bps, capital_base=CAPITAL_BASE)
            summary_rows.append(performance_metrics(rets, equity, weights, trades, framework=framework, variant=variant, top_k=top_k, cost_bps=cost_bps))

trade_log = pd.concat(trade_logs, ignore_index=True) if trade_logs else pd.DataFrame()
backtest_summary = pd.DataFrame(summary_rows).sort_values(['framework', 'variant', 'top_k']).reset_index(drop=True)
metric_cols = ['framework', 'variant', 'top_k', 'cost_bps', 'trades', 'final_equity', 'total_return', 'annualized_return', 'annualized_vol', 'sharpe', 'max_drawdown', 'avg_gross_exposure', 'avg_net_exposure', 'fully_invested_days', 'cash_days']
display(backtest_summary[metric_cols])
display(trade_log.groupby(['variant', 'top_k', 'action'], as_index=False).size().head(50))

,framework,variant,top_k,cost_bps,trades,final_equity,total_return,annualized_return,annualized_vol,sharpe,max_drawdown,avg_gross_exposure,avg_net_exposure,fully_invested_days,cash_days
0,backtesting_py_shared_book,long_only,5,6.0000,7,"2,720,918.2210",1.7517,0.1697,0.2223,0.8088,-0.2827,1.0000,1.0000,1.0000,0.0000
1,backtesting_py_shared_book,long_only,10,6.0000,246,"5,610,868.6175",4.6578,0.3079,0.2440,1.2173,-0.3709,0.9622,0.9622,0.7087,0.0000
2,backtesting_py_shared_book,long_only,20,6.0000,364,"2,808,178.6368",1.8199,0.1742,0.1338,1.2625,-0.2056,0.5010,0.5010,0.0000,0.0000
3,backtesting_py_shared_book,long_only,40,6.0000,364,"1,700,232.9049",0.7038,0.0860,0.0669,1.2625,-0.1056,0.2505,0.2505,0.0000,0.0000
4,backtesting_py_shared_book,long_short,5,6.0000,17,"4,103,309.0975",3.1183,0.2451,0.2044,1.1720,-0.3041,1.0000,0.9304,1.0000,0.0000
5,backtesting_py_shared_book,long_short,10,6.0000,84,"4,562,957.4885",3.5828,0.2659,0.2042,1.2535,-0.3053,1.0000,0.7389,1.0000,0.0000
6,backtesting_py_shared_book,long_short,20,6.0000,725,"1,945,753.4371",0.9521,0.1092,0.1103,0.9901,-0.1824,0.6500,0.3519,0.0000,0.0000
7,backtesting_py_shared_book,long_short,40,6.0000,725,"1,408,626.0067",0.4109,0.0548,0.0551,0.9901,-0.0939,0.3250,0.1760,0.0000,0.0000
8,backtesting_py_shared_book,short_only,5,6.0000,341,"185,770.6180",-0.8149,-0.2299,0.2326,-1.0043,-0.8549,0.5936,-0.5936,0.0738,0.0191
9,backtesting_py_shared_book,short_only,10,6.0000,361,"442,516.1035",-0.5583,-0.1189,0.1164,-1.0263,-0.6075,0.2981,-0.2981,0.0000,0.0191


,variant,top_k,action,size
0,long_only,5,enter_long,6
1,long_only,5,exit_long,1
2,long_only,10,enter_long,128
3,long_only,10,exit_long,118
4,long_only,20,enter_long,187
5,long_only,20,exit_long,177
6,long_only,40,enter_long,187
7,long_only,40,exit_long,177
8,long_short,5,enter_long,7
9,long_short,5,enter_short,4


## Model vs Trading Comparison

In [12]:
model_oos_summary = model_results.loc[model_results['status'].eq('ok'), ['source', 'family', 'oos_rows', 'oos_accuracy', 'oos_balanced_accuracy', 'oos_macro_f1']].copy()
display(model_oos_summary.sort_values('oos_macro_f1', ascending=False))

framework_comparison = (
    backtest_summary.groupby(['variant', 'top_k'], as_index=False)
    .agg(
        best_total_return=('total_return', 'max'),
        worst_total_return=('total_return', 'min'),
        avg_total_return=('total_return', 'mean'),
        best_sharpe=('sharpe', 'max'),
        worst_sharpe=('sharpe', 'min'),
        avg_gross_exposure=('avg_gross_exposure', 'mean'),
        avg_net_exposure=('avg_net_exposure', 'mean'),
    )
    .sort_values(['best_sharpe', 'best_total_return'], ascending=False)
)
display(framework_comparison)

,source,family,oos_rows,oos_accuracy,oos_balanced_accuracy,oos_macro_f1
0,financetoolkit,ft_ratios_valuation,3759,0.4302,0.3953,0.3912
1,financetoolkit,ft_ratios_solvency,3759,0.2934,0.3448,0.2676
2,fmp,fmp_cash_mcap,3759,0.2969,0.3642,0.2555
3,financetoolkit,ft_ratios_efficiency,3759,0.2854,0.3410,0.2515
4,financetoolkit,ft_ratios_profitability,3759,0.2825,0.3402,0.2496
5,fmp,fmp_daily_ev_yield,3759,0.2876,0.3533,0.2486
6,fmp,fmp_daily_mcap_yield,3759,0.2961,0.3635,0.2480
7,fmp,fmp_balance_mcap,3759,0.2985,0.3607,0.2423
8,fmp,fmp_income_mcap,3759,0.2966,0.3588,0.2413
9,fmp,fmp_daily_ev_multiple,3759,0.2857,0.3524,0.2413


,variant,top_k,best_total_return,worst_total_return,avg_total_return,best_sharpe,worst_sharpe,avg_gross_exposure,avg_net_exposure
2,long_only,20,1.8224,1.8074,1.8166,1.2635,1.2572,0.5010,0.5010
3,long_only,40,0.7045,0.7000,0.7028,1.2635,1.2572,0.2505,0.2505
5,long_short,10,3.5845,3.5744,3.5806,1.2538,1.2519,1.0000,0.7389
1,long_only,10,4.6646,4.6243,4.6489,1.2181,1.2134,0.9622,0.9622
4,long_short,5,3.1188,3.1159,3.1177,1.1722,1.1714,1.0000,0.9304
6,long_short,20,0.9555,0.9348,0.9475,0.9926,0.9774,0.6500,0.3519
7,long_short,40,0.4122,0.4046,0.4092,0.9926,0.9774,0.3250,0.1760
0,long_only,5,1.7518,1.7515,1.7517,0.8088,0.8085,1.0000,1.0000
8,short_only,5,-0.8143,-0.8180,-0.8157,-1.0021,-1.0157,0.5936,-0.5936
11,short_only,40,-0.1777,-0.1799,-0.1785,-1.0239,-1.0383,0.0745,-0.0745


In [13]:
analysis_lines = [
    '## Written Analysis',
    '',
    f'- Universe: {len(symbols)} FMP 1T+ symbols; {len(event_symbols)} had event coverage.',
    f'- Training window: all available rows through {TRAIN_END.date()}. Out-of-sample model and trading window starts {OOS_START.date()}.',
    f'- Trained GPU RF feature-family models: {len(models)}.',
    f'- OOS prediction rows: {len(mean_scores):,} across {mean_scores["symbol"].nunique()} symbols and {mean_scores["date"].nunique()} dates.',
    f'- Strategy variants: long_only, short_only, long_short with top_k={TOP_K_VALUES}; trade size is fixed at 1/top_k and unused slots stay cash.',
]
if not model_oos_summary.empty:
    best_model = model_oos_summary.sort_values('oos_macro_f1', ascending=False).iloc[0]
    analysis_lines.append(f'- Best OOS classifier family: {best_model["source"]}.{best_model["family"]} with macro_f1={best_model["oos_macro_f1"]:.4f}.')
if not backtest_summary.empty:
    best_trade = backtest_summary.sort_values(['sharpe', 'total_return'], ascending=False).iloc[0]
    analysis_lines.extend([
        '',
        'Best trading row by Sharpe:',
        f'- {best_trade["framework"]} {best_trade["variant"]} top_k={int(best_trade["top_k"])}: total_return={best_trade["total_return"]:.2%}, sharpe={best_trade["sharpe"]:.2f}, max_drawdown={best_trade["max_drawdown"]:.2%}, avg_gross={best_trade["avg_gross_exposure"]:.2f}, avg_net={best_trade["avg_net_exposure"]:.2f}.',
        '',
        'Interpretation:',
        '- This notebook validates the shared-book trading policy; it does not reuse the old single-symbol sleeve runners.',
        '- Framework rows use the same target weights and differ only by realistic cost assumptions, because the existing native adapters are not shared-book compatible yet.',
        '- If model OOS macro F1 is weak but trading performance is strong, the ranking/threshold policy may matter more than multiclass accuracy. If both are weak, the collapsed labels are probably not enough for this universe.',
    ])
from IPython.display import Markdown, display
display(Markdown('\n'.join(analysis_lines)))

## Written Analysis

- Universe: 14 FMP 1T+ symbols; 13 had event coverage.
- Training window: all available rows through 2019-12-31. Out-of-sample model and trading window starts 2020-01-01.
- Trained GPU RF feature-family models: 15.
- OOS prediction rows: 21,151 across 13 symbols and 1627 dates.
- Strategy variants: long_only, short_only, long_short with top_k=[5, 10, 20, 40]; trade size is fixed at 1/top_k and unused slots stay cash.
- Best OOS classifier family: financetoolkit.ft_ratios_valuation with macro_f1=0.3912.

Best trading row by Sharpe:
- zipline_shared_book long_only top_k=20: total_return=182.24%, sharpe=1.26, max_drawdown=-20.56%, avg_gross=0.50, avg_net=0.50.

Interpretation:
- This notebook validates the shared-book trading policy; it does not reuse the old single-symbol sleeve runners.
- Framework rows use the same target weights and differ only by realistic cost assumptions, because the existing native adapters are not shared-book compatible yet.
- If model OOS macro F1 is weak but trading performance is strong, the ranking/threshold policy may matter more than multiclass accuracy. If both are weak, the collapsed labels are probably not enough for this universe.

## Written Analysis

- Universe: 14 FMP 1T+ symbols; 13 had event coverage.
- Training window: all available rows through 2019-12-31. Out-of-sample model and trading window starts 2020-01-01.
- Trained GPU RF feature-family models: 15.
- OOS prediction rows: 21,151 across 13 symbols and 1627 dates.
- Strategy variants: long_only, short_only, long_short with top_k=[5, 10, 20, 40]; trade size is fixed at 1/top_k and unused slots stay cash.
- Best OOS classifier family: financetoolkit.ft_ratios_valuation with macro_f1=0.3912.

Best trading row by Sharpe:
- zipline_shared_book long_only top_k=20: total_return=182.24%, sharpe=1.26, max_drawdown=-20.56%, avg_gross=0.50, avg_net=0.50.

Interpretation:
- This notebook validates the shared-book trading policy; it does not reuse the old single-symbol sleeve runners.
- Framework rows use the same target weights and differ only by realistic cost assumptions, because the existing native adapters are not shared-book compatible yet.
- If model OOS macro F1 is weak but trading performance is strong, the ranking/threshold policy may matter more than multiclass accuracy. If both are weak, the collapsed labels are probably not enough for this universe.